In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import catboost, lightgbm, xgboost
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, log_loss
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path("../data")

In [2]:
items = pd.read_csv(DATA_DIR / "items.csv")
terms = pd.read_csv(DATA_DIR / "terms.csv")
training_pairs = pd.read_csv(DATA_DIR / "training_pairs.csv")
submission_pairs = pd.read_csv(DATA_DIR / "submission_pairs.csv")

print(f"terms={len(terms):,}")
print(f"items={len(items):,}")
print(f"training pairs={len(training_pairs):,}")
print(f"submission pairs={len(submission_pairs):,}")

terms=50,153
items=962,873
training pairs=250,000
submission pairs=3,359,679


In [3]:
df = submission_pairs.merge(terms, on="term_id", how="left").merge(items, on="item_id", how="left")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3359679 entries, 0 to 3359678
Data columns (total 10 columns):
 #   Column      Dtype
---  ------      -----
 0   id          str  
 1   term_id     str  
 2   item_id     str  
 3   query       str  
 4   title       str  
 5   category    str  
 6   brand       str  
 7   gender      str  
 8   age_group   str  
 9   attributes  str  
dtypes: str(10)
memory usage: 256.3 MB


In [5]:
df["gender"].nunique()

4

In [41]:
items["attributes"].eq("unknown").sum()

np.int64(19025)

In [35]:
print(df["gender"].eq("unknown").sum())


2415941


In [32]:
df["age_group"].eq("unknown").sum()

np.int64(2325982)

In [29]:
df.head()

,id,term_id,item_id,query,title,category,brand,gender,age_group,attributes
0,TST_8ede5e2b0176ec,TERM_194bea06,ITEM_41f4c31b7c05,fono yayınları,ilk 100 sözcük flash cards türkçe - ingilizce ...,kitap/çocuk ve ebeveyn kitapları/aktivite & eğ...,circle toys,unknown,unknown,"boyut: normal boy, basım dili: ingilizce, yaza..."
1,TST_2a09d773d97e6e,TERM_eacb5395,ITEM_974f6e875b57,luck,so fever him,kozmetik & kişisel bakım/parfüm ve deodorant/p...,oriflame,erkek,yetişkin,"tip: edt, koku türü: fresh, refill: hayır, hac..."
2,TST_a4a86755c99d3d,TERM_07ab54b2,ITEM_d81d95b30383,ayakkabi prensi klasik topuklu ayakkabı,kadın kırmızı hakiki deri 34-35-41-42 numara t...,ayakkabı/topuklu ayakkabı/klasik topuklu ayakkabı,tunaelli,kadın,yetişkin,"materyal: hakiki deri, sürdürülebilirlik detay..."
3,TST_e2db221eaae0ec,TERM_80f9a29f,ITEM_027ce477fcac,pandora gold,"can gazetelikli c yan sehpa, tekerlekli sehpa ...",ev & mobilya/mobilya/çalışma odası/dergilik & ...,modilayn,unknown,unknown,"materyal: ahşap, boyut/ebat: tek ebat, stil: r..."
4,TST_2718f17b96c6f5,TERM_2b89cd08,ITEM_b0897b7143fa,uzun tesettür trençkot,boş cam şişe ambalaj 5 ml rolon 50 adet takım,banyo yapı & hırdavat/hırdavat/genel bakım spreyi,paradise,unknown,unknown,paket içeriği: 50'li


In [15]:
items.head()

,item_id,title,category,brand,gender,age_group,attributes
0,ITEM_3a515fb7125d,erkek kumaş usb kulaklık çıkışlı bodybag göğüs...,aksesuar/çanta/omuz çantası,newish polo,erkek,yetişkin,"materyal: tekstil, deri kalitesi: parça mevcut..."
1,ITEM_94fe5db929cd,orijinal italyan charm bileklik kişiselleştiri...,aksesuar/takı & mücevher/bileklik/çelik bileklik,nomination,unisex,yetişkin,"taş cinsi: yok, renk: gri, materyal: paslanmaz..."
2,ITEM_576afa5db9a1,kupa bardak alchemist high,ev & mobilya/ev/sofra & mutfak/sofra/bardak,qivi,unknown,unknown,"materyal: seramik, parça sayısı: 1, renk: beya..."
3,ITEM_846531067083,çizgili geniş kol koton gömlek,giyim/üst giyim/gömlek,ayhan,kadın,yetişkin,"desen: çizgili, kalıp: regular, yaka tipi: v y..."
4,ITEM_1c3943c4d220,kapaklı katlanabilir puantiyeli pembe 60 litre...,ev & mobilya/ev/ev gereçleri/çamaşırlık ev ger...,serstil,unknown,unknown,"materyal: polipropilen, renk: pembe, hacim: 60..."


In [17]:
terms.head()

,term_id,query
0,TERM_f2b61db2,defacto kız bebek elbise
1,TERM_00247623,bebek oto koltuğu
2,TERM_c977024b,sava lastik
3,TERM_dbdbddbb,kadın tesettür kışlık gömlek
4,TERM_c06329e3,erkek çocuk kranpon


In [16]:
training_pairs.head()

,id,term_id,item_id,label
0,TRN_c639ed31a5,TERM_68c2d117,ITEM_0c59058c3908,1
1,TRN_ca3ce092f7,TERM_3374e63e,ITEM_d0d66d79750c,1
2,TRN_d4e3637fd6,TERM_a2d68022,ITEM_cd5152c2807c,1
3,TRN_eaaa1cecbb,TERM_161ac15a,ITEM_f1ebadc41d01,1
4,TRN_42ef8efcad,TERM_6331588a,ITEM_e7a0765b4788,1
